# Busca Híbrida com Reciprocal Rank Fusion (RRF)

Este notebook reproduz, em miniatura e com um exemplo genérico, a arquitetura de busca usada em sistemas de RAG em produção que combinam **busca semântica (vetorial)** com **busca lexical (BM25)**, unindo os dois rankings através de **Reciprocal Rank Fusion (RRF)**.

A ideia central é que os dois métodos erram em lugares diferentes:
- A **busca semântica** entende sinônimos e paráfrases, mas pode "diluir" termos exatos (códigos de erro, números, siglas) no espaço vetorial.
- A **busca lexical (BM25)** é excelente para termos exatos, mas não entende que "app trava" e "aplicativo travando" são a mesma coisa se as palavras não coincidem.

O RRF combina os **rankings** (não os scores brutos, que têm escalas diferentes) das duas buscas, permitindo que o melhor dos dois mundos apareça no topo.

> Base de conhecimento usada aqui é fictícia (suporte técnico de um produto SaaS genérico), construída apenas para ilustrar a técnica.

In [1]:
!pip install -qU FlagEmbedding rank_bm25 nltk langchain-text-splitters numpy scikit-learn

In [2]:
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity
from langchain_text_splitters import RecursiveCharacterTextSplitter

try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords", quiet=True)
    nltk.download("punkt_tab", quiet=True)

STOPWORDS_PT = set(stopwords.words("portuguese"))

/home/ryan-mazzeu/Desktop/TCC/TCC/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Base de conhecimento (exemplo genérico)

Cada entrada representa um "documento" de uma base de conhecimento de suporte técnico. Alguns têm termos exatos importantes (códigos de erro, números, siglas), outros são mais conceituais — de propósito, para depois compararmos onde cada método de busca se sai melhor.

In [3]:
documentos_fonte = {
    "auth": (
        "Para redefinir sua senha, acesse a tela de login e clique em 'Esqueci minha senha'. "
        "Um link de redefinição é enviado por e-mail e expira em 30 minutos. "
        "A autenticação de dois fatores (2FA) pode ser ativada em Configurações > Segurança "
        "usando qualquer aplicativo autenticador compatível com TOTP."
    ),
    "rate_limit": (
        "A API aplica limite de taxa de 100 requisições por minuto por chave de API. "
        "Quando o limite é excedido, o servidor retorna o código de status HTTP 429. "
        "O cabeçalho de resposta XRateLimitRemaining indica quantas requisições ainda "
        "podem ser feitas na janela atual antes do bloqueio temporário."
    ),
    "webhooks": (
        "Webhooks que falham no primeiro envio são reenviados automaticamente até 5 vezes, "
        "com backoff exponencial entre as tentativas. Após 5 tentativas sem sucesso, o evento "
        "é marcado como falho e uma notificação é enviada ao administrador da conta."
    ),
    "billing": (
        "As faturas são geradas no primeiro dia útil de cada mês e enviadas por e-mail em "
        "formato PDF. Cobranças recusadas pelo cartão resultam em uma nova tentativa "
        "automática após 3 dias corridos."
    ),
    "export": (
        "É possível exportar todos os dados da conta em formato CSV através da seção "
        "Configurações > Exportar Dados. O arquivo gerado é enviado por e-mail em até 24 horas."
    ),
    "sla": (
        "O acordo de nível de serviço garante disponibilidade de 99.9% ao mês. "
        "Períodos de manutenção programada são comunicados com 48 horas de antecedência "
        "e não contam para o cálculo de indisponibilidade."
    ),
    "error_db": (
        "O erro E4042 indica timeout na conexão com o banco de dados durante a execução "
        "de uma consulta. Geralmente ocorre em picos de tráfego e é resolvido automaticamente "
        "pelo sistema de retry em poucos segundos."
    ),
    "email_deliv": (
        "Para melhorar a entregabilidade de e-mails transacionais, configure os registros "
        "SPF e DKIM no DNS do seu domínio. Sem essa configuração, as mensagens podem ser "
        "classificadas como spam pelos provedores de e-mail dos destinatários."
    ),
    "lgpd": (
        "Usuários podem solicitar a exclusão completa de seus dados pessoais a qualquer "
        "momento, em conformidade com a LGPD. A solicitação é processada em até 15 dias "
        "úteis e, uma vez concluída, é irreversível."
    ),
    "mobile_crash": (
        "Alguns usuários relataram travamentos do aplicativo móvel logo após atualizar "
        "para o iOS 17. O problema foi identificado como incompatibilidade com uma versão "
        "antiga do SDK interno e já foi corrigido na versão 4.2.1 do aplicativo."
    ),
    "plan_change": (
        "Upgrades de plano são aplicados imediatamente e cobrados de forma proporcional "
        "aos dias restantes do ciclo. Downgrades de plano só entram em vigor no início "
        "do próximo ciclo de faturamento."
    ),
    "performance": (
        "Se o carregamento de relatórios grandes estiver lento, recomendamos habilitar "
        "a paginação de resultados nas configurações avançadas, o que reduz "
        "significativamente o tempo de resposta percebido pelo usuário."
    ),
}

print(f"{len(documentos_fonte)} documentos na base de conhecimento.")

12 documentos na base de conhecimento.


## 2. Chunking

Cada documento é dividido em chunks menores, como em qualquer pipeline de ingestão para RAG. Guardamos `document_id` e `chunk_index` no metadado de cada chunk — são esses campos que depois permitem casar um mesmo chunk encontrado pelos dois métodos de busca.

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=220,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []  # cada item: {"document_id", "chunk_index", "text"}

for doc_id, texto in documentos_fonte.items():
    partes = text_splitter.split_text(texto)
    for i, parte in enumerate(partes):
        chunks.append({"document_id": doc_id, "chunk_index": i, "text": parte})

print(f"Total de chunks gerados: {len(chunks)}\n")
for c in chunks:
    print(f"[{c['document_id']}#{c['chunk_index']}] {c['text']}")

Total de chunks gerados: 17

[auth#0] Para redefinir sua senha, acesse a tela de login e clique em 'Esqueci minha senha'. Um link de redefinição é enviado por e-mail e expira em 30 minutos
[auth#1] . A autenticação de dois fatores (2FA) pode ser ativada em Configurações > Segurança usando qualquer aplicativo autenticador compatível com TOTP.
[rate_limit#0] A API aplica limite de taxa de 100 requisições por minuto por chave de API. Quando o limite é excedido, o servidor retorna o código de status HTTP 429
[rate_limit#1] . O cabeçalho de resposta XRateLimitRemaining indica quantas requisições ainda podem ser feitas na janela atual antes do bloqueio temporário.
[webhooks#0] Webhooks que falham no primeiro envio são reenviados automaticamente até 5 vezes, com backoff exponencial entre as tentativas
[webhooks#1] . Após 5 tentativas sem sucesso, o evento é marcado como falho e uma notificação é enviada ao administrador da conta.
[billing#0] As faturas são geradas no primeiro dia útil de cada

## 3. Busca semântica (vetorial)

Geramos embeddings densos para todos os chunks com o BGE-M3 (mesmo modelo do notebook anterior) e comparamos a query por similaridade de cosseno — exatamente o que um banco vetorial faz internamente (ex.: operador `<=>` do pgvector é uma distância de cosseno; `score = 1 - distância`).

In [5]:
from FlagEmbedding import BGEM3FlagModel

modelo_embedding = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)

textos_chunks = [c["text"] for c in chunks]
vetores_documentos = modelo_embedding.encode(
    textos_chunks,
    batch_size=12,
    max_length=512,
    return_dense=True,
    return_sparse=False,
    return_colbert_vecs=False,
)["dense_vecs"]

print(f"Vetores gerados: {vetores_documentos.shape[0]} x {vetores_documentos.shape[1]} dimensões")

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Fetching 30 files: 100%|██████████| 30/30 [00:00<00:00, 208671.84it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   1%|▏         | 5/391 [00:00<00:13, 28.40it/s]

Loading weights:  63%|██████▎   | 245/391 [00:00<00:00, 1042.42it/s]

...

Inference Embeddings:   0%|          | 0/2 [00:00<?, ?it/s]

Inference Embeddings:  50%|█████     | 1/2 [00:01<00:01,  1.37s/it]

Inference Embeddings: 100%|██████████| 2/2 [00:01<00:00,  1.21it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:01<00:00,  1.10it/s]

Vetores gerados: 17 x 1024 dimensões


In [6]:
def busca_semantica(query, k=5):
    """Busca vetorial: embedding da query + cosine similarity contra todos os chunks."""
    vetor_query = modelo_embedding.encode([query], return_dense=True)["dense_vecs"]
    scores = cosine_similarity(vetor_query, vetores_documentos)[0]

    ranking = np.argsort(scores)[::-1][:k]
    return [(chunks[i], float(scores[i])) for i in ranking]

## 4. Busca lexical (BM25)

BM25 é um algoritmo clássico de ranking textual (usado por padrão em motores como Elasticsearch/Postgres FTS). Diferente da busca semântica, ele pontua com base na **frequência exata de termos**, ponderada pela raridade do termo no corpus — por isso é tão bom para números, códigos e siglas exatas.

In [7]:
def tokenizar(texto):
    """Tokeniza, remove stopwords e termos muito curtos — igual ao pré-processamento usual antes do BM25."""
    tokens = word_tokenize(texto.lower(), language="portuguese")
    return [t for t in tokens if t.isalnum() and t not in STOPWORDS_PT and len(t) > 2]

corpus_tokenizado = [tokenizar(c["text"]) for c in chunks]
bm25 = BM25Okapi(corpus_tokenizado, k1=1.0, b=0.75)


def busca_bm25(query, k=5):
    """Busca lexical: score BM25 da query tokenizada contra todos os chunks."""
    tokens_query = tokenizar(query)
    scores = bm25.get_scores(tokens_query)

    ranking = np.argsort(scores)[::-1][:k]
    return [(chunks[i], float(scores[i])) for i in ranking]

## 5. Fusão RRF (Reciprocal Rank Fusion)

Scores de cosine similarity (0 a 1) e scores de BM25 (sem limite superior, depende do corpus) não são comparáveis diretamente. O RRF resolve isso ignorando o *valor* do score e usando apenas a **posição no ranking** de cada método:

$$\text{RRF}(d) = \sum_{\text{método}} \frac{\text{peso}_{\text{método}}}{k + \text{rank}_{\text{método}}(d)}$$

Um chunk que aparece bem ranqueado nos dois métodos acumula pontuação dos dois; um chunk que só um método encontrou ainda pode subir se ficou muito bem posicionado nesse método. `k` (tipicamente 60) suaviza a curva para que a diferença entre a 1ª e a 2ª posição não domine demais o resultado.

In [8]:
def reciprocal_rank_fusion(
    resultados_semanticos,
    resultados_bm25,
    rrf_k=60,
    peso_semantico=0.25,
    peso_bm25=0.75,
):
    """Combina dois rankings independentes em um único ranking fundido."""

    def chunk_key(chunk):
        return (chunk["document_id"], chunk["chunk_index"])

    rrf_scores = {}
    detalhes = {}

    for rank, (chunk, score) in enumerate(resultados_semanticos):
        key = chunk_key(chunk)
        rrf_scores[key] = rrf_scores.get(key, 0.0) + peso_semantico / (rrf_k + rank + 1)
        detalhes.setdefault(key, {"chunk": chunk})
        detalhes[key]["semantic_score"] = score
        detalhes[key]["semantic_rank"] = rank + 1

    for rank, (chunk, score) in enumerate(resultados_bm25):
        key = chunk_key(chunk)
        rrf_scores[key] = rrf_scores.get(key, 0.0) + peso_bm25 / (rrf_k + rank + 1)
        detalhes.setdefault(key, {"chunk": chunk})
        detalhes[key]["bm25_score"] = score
        detalhes[key]["bm25_rank"] = rank + 1

    ranking_final = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)

    return [
        {
            "chunk": detalhes[key]["chunk"],
            "rrf_score": rrf_score,
            "semantic_rank": detalhes[key].get("semantic_rank"),
            "bm25_rank": detalhes[key].get("bm25_rank"),
        }
        for key, rrf_score in ranking_final
    ]

## 6. Pipeline completo: busca híbrida

Junta os três passos: busca semântica com um `candidate_k` maior que o `k` final (para dar mais material à fusão), busca BM25, fusão RRF e normalização do score final (0 a 1, relativo ao melhor score do próprio batch).

In [9]:
def busca_hibrida(query, k=5, candidate_k=10, peso_semantico=0.25, peso_bm25=0.75):
    semanticos = busca_semantica(query, k=candidate_k)
    lexicais = busca_bm25(query, k=candidate_k)

    fundidos = reciprocal_rank_fusion(
        semanticos, lexicais, peso_semantico=peso_semantico, peso_bm25=peso_bm25
    )[:k]

    maior_score = max((r["rrf_score"] for r in fundidos), default=1.0) or 1.0
    for r in fundidos:
        r["score_normalizado"] = round(r["rrf_score"] / maior_score, 4)

    return fundidos

## 7. Comparando: Semântica vs BM25 vs Híbrida (RRF)

Testamos quatro perguntas escolhidas de propósito:
- Uma com **número exato** (favorece BM25).
- Uma com **paráfrase forte**, sem palavras em comum com o texto original (favorece a busca semântica).
- Uma **mista**, que se beneficia dos dois.
- Uma **ambígua/curta**, para ver como cada método se comporta com pouco sinal.

In [10]:
def mostrar_resultado(chunk, extra=""):
    print(f"   [{chunk['document_id']}#{chunk['chunk_index']}] {chunk['text'][:100]}...{extra}")


perguntas_teste = [
    "limite de 100 requisições por minuto estourou",   # número exato -> BM25 deve brilhar
    "meu app trava depois que atualizei o iPhone",      # paráfrase de "iOS 17" -> semântica deve brilhar
    "o que significa o erro E4042?",                    # código exato -> BM25 deve brilhar
    "quero cancelar tudo e apagar meus dados",          # paráfrase de LGPD -> semântica deve brilhar
]

for pergunta in perguntas_teste:
    print("=" * 100)
    print(f"PERGUNTA: {pergunta}\n")

    print("-- Top 1 Semântica --")
    mostrar_resultado(busca_semantica(pergunta, k=1)[0][0])

    print("-- Top 1 BM25 --")
    mostrar_resultado(busca_bm25(pergunta, k=1)[0][0])

    print("-- Top 3 Híbrida (RRF) --")
    for r in busca_hibrida(pergunta, k=3):
        mostrar_resultado(
            r["chunk"],
            extra=f"  [rrf={r['score_normalizado']}, rank_sem={r['semantic_rank']}, rank_bm25={r['bm25_rank']}]",
        )
    print()

PERGUNTA: limite de 100 requisições por minuto estourou

-- Top 1 Semântica --


   [rate_limit#0] A API aplica limite de taxa de 100 requisições por minuto por chave de API. Quando o limite é excedi...
-- Top 1 BM25 --
   [rate_limit#0] A API aplica limite de taxa de 100 requisições por minuto por chave de API. Quando o limite é excedi...
-- Top 3 Híbrida (RRF) --


   [rate_limit#0] A API aplica limite de taxa de 100 requisições por minuto por chave de API. Quando o limite é excedi...  [rrf=1.0, rank_sem=1, rank_bm25=1]
   [rate_limit#1] . O cabeçalho de resposta XRateLimitRemaining indica quantas requisições ainda podem ser feitas na j...  [rrf=0.9839, rank_sem=2, rank_bm25=2]
   [performance#0] Se o carregamento de relatórios grandes estiver lento, recomendamos habilitar a paginação de resulta...  [rrf=0.9505, rank_sem=8, rank_bm25=3]

PERGUNTA: meu app trava depois que atualizei o iPhone

-- Top 1 Semântica --


   [mobile_crash#0] Alguns usuários relataram travamentos do aplicativo móvel logo após atualizar para o iOS 17...
-- Top 1 BM25 --
   [performance#0] Se o carregamento de relatórios grandes estiver lento, recomendamos habilitar a paginação de resulta...
-- Top 3 Híbrida (RRF) --


   [performance#0] Se o carregamento de relatórios grandes estiver lento, recomendamos habilitar a paginação de resulta...  [rrf=1.0, rank_sem=4, rank_bm25=1]
   [mobile_crash#1] . O problema foi identificado como incompatibilidade com uma versão antiga do SDK interno e já foi c...  [rrf=0.9837, rank_sem=2, rank_bm25=3]
   [plan_change#0] Upgrades de plano são aplicados imediatamente e cobrados de forma proporcional aos dias restantes do...  [rrf=0.977, rank_sem=7, rank_bm25=2]

PERGUNTA: o que significa o erro E4042?

-- Top 1 Semântica --


   [error_db#0] O erro E4042 indica timeout na conexão com o banco de dados durante a execução de uma consulta. Gera...
-- Top 1 BM25 --
   [error_db#0] O erro E4042 indica timeout na conexão com o banco de dados durante a execução de uma consulta. Gera...
-- Top 3 Híbrida (RRF) --


   [error_db#0] O erro E4042 indica timeout na conexão com o banco de dados durante a execução de uma consulta. Gera...  [rrf=1.0, rank_sem=1, rank_bm25=1]
   [performance#0] Se o carregamento de relatórios grandes estiver lento, recomendamos habilitar a paginação de resulta...  [rrf=0.9622, rank_sem=8, rank_bm25=2]
   [mobile_crash#1] . O problema foi identificado como incompatibilidade com uma versão antiga do SDK interno e já foi c...  [rrf=0.9498, rank_sem=2, rank_bm25=5]

PERGUNTA: quero cancelar tudo e apagar meus dados

-- Top 1 Semântica --


   [lgpd#0] Usuários podem solicitar a exclusão completa de seus dados pessoais a qualquer momento, em conformid...
-- Top 1 BM25 --
   [export#0] É possível exportar todos os dados da conta em formato CSV através da seção Configurações > Exportar...
-- Top 3 Híbrida (RRF) --


   [export#0] É possível exportar todos os dados da conta em formato CSV através da seção Configurações > Exportar...  [rrf=1.0, rank_sem=2, rank_bm25=1]
   [lgpd#0] Usuários podem solicitar a exclusão completa de seus dados pessoais a qualquer momento, em conformid...  [rrf=0.9919, rank_sem=1, rank_bm25=2]
   [error_db#0] O erro E4042 indica timeout na conexão com o banco de dados durante a execução de uma consulta. Gera...  [rrf=0.951, rank_sem=9, rank_bm25=3]



## 8. O peso importa: o RRF não é uma bala de prata

Olhando os resultados acima com atenção: nas duas perguntas com **números e códigos exatos** ("100 requisições", "E4042"), semântica e BM25 já concordavam no 1º lugar, então o RRF só confirmou.

Mas nas duas perguntas **parafraseadas** ("iPhone" → doc fala em "iOS 17"; "apagar meus dados" → doc fala em "exclusão... LGPD"), a busca semântica acertou o chunk certo em 1º lugar — e o RRF **errou mesmo assim**, porque os pesos usados (25% semântica / 75% BM25, os mesmos da arquitetura original) dão tanto peso ao ranking do BM25 que um acerto forte da semântica não é suficiente para superar um acerto fraco do BM25 em outro chunk.

Ou seja: **o RRF não corrige automaticamente as fraquezas de cada método — ele só faz uma média ponderada de rankings**. Se o peso estiver desbalanceado para o tipo de pergunta que os usuários realmente fazem, o híbrido pode herdar o erro do método mais pesado. Vamos comprovar isso re-rodando as duas perguntas parafraseadas com pesos diferentes.

In [11]:
perguntas_parafraseadas = [
    "meu app trava depois que atualizei o iPhone",
    "quero cancelar tudo e apagar meus dados",
]

configuracoes_peso = [
    ("Produção (25% sem. / 75% bm25)", 0.25, 0.75),
    ("Equilibrado (50% / 50%)", 0.50, 0.50),
    ("Semântica dominante (75% / 25%)", 0.75, 0.25),
]

for pergunta in perguntas_parafraseadas:
    print("=" * 100)
    print(f"PERGUNTA: {pergunta}\n")
    for nome, peso_sem, peso_bm25 in configuracoes_peso:
        top1 = busca_hibrida(pergunta, k=1, peso_semantico=peso_sem, peso_bm25=peso_bm25)[0]
        chunk = top1["chunk"]
        print(f"  {nome:35s} -> [{chunk['document_id']}#{chunk['chunk_index']}] {chunk['text'][:70]}...")
    print()

PERGUNTA: meu app trava depois que atualizei o iPhone



  Produção (25% sem. / 75% bm25)      -> [performance#0] Se o carregamento de relatórios grandes estiver lento, recomendamos ha...


  Equilibrado (50% / 50%)             -> [mobile_crash#0] Alguns usuários relataram travamentos do aplicativo móvel logo após at...


  Semântica dominante (75% / 25%)     -> [mobile_crash#0] Alguns usuários relataram travamentos do aplicativo móvel logo após at...

PERGUNTA: quero cancelar tudo e apagar meus dados



  Produção (25% sem. / 75% bm25)      -> [export#0] É possível exportar todos os dados da conta em formato CSV através da ...


  Equilibrado (50% / 50%)             -> [lgpd#0] Usuários podem solicitar a exclusão completa de seus dados pessoais a ...


  Semântica dominante (75% / 25%)     -> [lgpd#0] Usuários podem solicitar a exclusão completa de seus dados pessoais a ...



## Conclusão

- A busca **semântica** venceu nas perguntas parafraseadas ("iPhone" → "iOS 17", "apagar meus dados" → "exclusão... LGPD"), porque entende o significado mesmo sem sobreposição de palavras.
- O **BM25** venceu nas perguntas com termos exatos (números, códigos como `E4042`), porque pondera fortemente a correspondência literal de tokens raros no corpus.
- A **fusão RRF com pesos de produção (25%/75%)** trouxe o chunk certo para o topo apenas quando os dois métodos já concordavam — nas duas perguntas parafraseadas, ela **manteve o erro do BM25**, porque o peso dado a ele era alto demais para o tipo de pergunta feita.
- Subindo o peso da semântica (50/50 ou 75/25) os dois casos parafraseados se corrigem, mas isso é uma troca: mais peso para semântica tende a piorar buscas por termos exatos (números, códigos), que dependem de correspondência literal.

**A lição prática para um TCC comparativo**: RRF é uma técnica de combinação, não uma garantia de acerto. A escolha dos pesos precisa refletir o perfil real das perguntas dos usuários (mais conceituais → mais peso semântico; mais buscas por termos/códigos exatos → mais peso BM25), e idealmente deveria ser validada empiricamente com métricas como Hit Rate e MRR sobre um conjunto de perguntas reais — exatamente como mencionado no notebook de busca semântica anterior.